# SQL-Mode Capability Showcase

This notebook demonstrates every Gremlin step that is supported in SQL-translation mode
(`/query/explain`), as documented in **Section 17** of `DESIGN_DOCUMENT.md`.

Each section shows:
- The Gremlin traversal
- The translated SQL (via `/query/explain`)
- Live results from native execution (via `/gremlin/query`)

**Prerequisites:**
1. Backend running: `mvn exec:java` (in repo root)
2. AML demo data loaded (run the setup cell below, or use `aml_demo_queries.ipynb` Step 2)

---

**Data model** (AML mapping):
```
Account --[TRANSFER]--> Account
Account --[BELONGS_TO]--> Bank
Account --[FLAGGED_BY]--> Alert
Account --[SENT_VIA]--> Country
Bank    --[LOCATED_IN]--> Country
```


In [1]:
import requests
import json as _json
import pandas as pd
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
MAPPING_PATH = next(
    (str(p) for p in [
        Path.cwd() / "mappings/aml-mapping.json",
        Path.cwd() / "demo/mappings/aml-mapping.json",
    ] if p.exists()),
    None,
)


def explain(gremlin: str) -> dict:
    """Call /query/explain and return the parsed JSON."""
    try:
        r = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin},
            timeout=10,
        )
        return r.json() if r.ok else {"error": f"HTTP {r.status_code}: {r.text}"}
    except Exception as e:
        return {"error": str(e)}


def run(gremlin: str) -> dict:
    """Call /gremlin/query and return the parsed JSON."""
    try:
        r = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": gremlin},
            timeout=30,
        )
        return r.json() if r.ok else {"error": f"HTTP {r.status_code}: {r.text}"}
    except Exception as e:
        return {"error": str(e)}


def show(gremlin: str, title: str = "", limit: int = 10):
    """Render Gremlin, SQL translation and live results side-by-side."""
    if title:
        display(Markdown(f"### {title}"))

    display(Markdown(f"**Gremlin:**\n```groovy\n{gremlin}\n```"))

    # SQL translation
    ex = explain(gremlin)
    if "error" in ex:
        display(Markdown(f"**SQL Translation:** *unavailable — {ex['error']}*"))
    else:
        sql = ex.get("translatedSql", "")
        params = ex.get("parameters", [])
        if sql:
            display(Markdown(f"**SQL:**\n```sql\n{sql}\n```"))
            if params:
                display(Markdown(f"**Parameters:** `{params}`"))
        else:
            display(Markdown("**SQL Translation:** *empty response*"))

    # Live results
    res = run(gremlin)
    if "error" in res:
        display(Markdown(f"**Runtime Error:** {res['error']}"))
        return

    rows = res.get("results", [])
    display(Markdown(f"**Result count:** {res.get('resultCount', len(rows))}"))
    if rows:
        sample = rows[:limit]
        if isinstance(sample[0], dict):
            display(pd.DataFrame(sample))
        else:
            for i, v in enumerate(sample, 1):
                print(f"{i}. {v}")


print("BASE_URL :", BASE_URL)
print("MAPPING  :", MAPPING_PATH)


BASE_URL : http://localhost:7000
MAPPING  : /Users/vjoshi/SourceCode/graph-query-engine/mappings/aml-mapping.json


## Setup — Health check, mapping upload & data load


In [2]:
# Health
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5).text
    provider = requests.get(f"{BASE_URL}/gremlin/provider", timeout=5).json().get("provider", "?")
    display(Markdown(f"**Health:** `{health}`    **Provider:** `{provider}`"))
except Exception as e:
    display(Markdown(f"**Backend not reachable:** {e}"))

# Mapping upload (idempotent)
if MAPPING_PATH:
    with open(MAPPING_PATH, "rb") as f:
        resp = requests.post(f"{BASE_URL}/mapping/upload", files={"file": f}, timeout=10)
    display(Markdown(f"**Mapping upload:** HTTP {resp.status_code} — {resp.text[:120]}"))
else:
    display(Markdown("**Mapping file not found.** Ensure `mappings/aml-mapping.json` exists."))


def _count(gremlin: str, default: int = 0) -> int:
    try:
        r = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": gremlin},
            timeout=10,
        ).json()
        vals = r.get("results") or []
        return int(vals[0]) if vals else default
    except Exception:
        return default


account_count = _count("g.V().hasLabel('Account').count()")
high_risk_count = _count("g.V().hasLabel('Account').has('riskScore', gt(70)).count()")
opened_missing_count = _count("g.V().hasLabel('Account').hasNot('openedDate').count()")
blocked_count = _count("g.V().hasLabel('Account').has('isBlocked','true').count()")
suspicious_unflagged_count = _count("g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).count()")

# Reload when dataset is empty OR appears to be legacy loader output.
needs_reload = account_count == 0 or (
    account_count > 0
    and high_risk_count == 0
    and opened_missing_count == 0
    and blocked_count == 0
    and suspicious_unflagged_count == 0
)

if needs_reload:
    reason = "empty graph" if account_count == 0 else "legacy dataset detected"
    display(Markdown(f"**Data loading:** {reason} — loading AML demo data..."))
    try:
        from demo.aml_csv_loader import AmlCsvLoader

        loader = AmlCsvLoader(base_url=BASE_URL)
        csv_path = next(
            (str(p) for p in [
                Path.cwd() / "demo/data/aml-demo.csv",
                Path.cwd() / "data/aml-demo.csv",
            ] if p.exists()),
            None,
        )
        if csv_path:
             stats = loader.load(csv_path, max_rows=100000, verbose=False)
             account_count = _count("g.V().hasLabel('Account').count()")
             high_risk_count = _count("g.V().hasLabel('Account').has('riskScore', gt(70)).count()")
             opened_missing_count = _count("g.V().hasLabel('Account').hasNot('openedDate').count()")
             blocked_count = _count("g.V().hasLabel('Account').has('isBlocked','true').count()")
             suspicious_unflagged_count = _count("g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).count()")
             display(Markdown(f"""✓ **Data loaded successfully:**
- Rows loaded: {stats.get('rowsLoaded', 0):,}
- Accounts created: {stats.get('accountsCreated', 0):,}
- Transfers created: {stats.get('transfersCreated', 0):,}
- Accounts in graph: {account_count:,}
- High risk (>70): {high_risk_count:,}
- Blocked accounts: {blocked_count:,}
- Missing openedDate: {opened_missing_count:,}
- Suspicious but unflagged: {suspicious_unflagged_count:,}"""))
        else:
            display(Markdown("✗ **CSV file not found.** Place `aml-demo.csv` in `demo/data/` or `data/` directory."))
    except ImportError:
        display(Markdown("⚠️ **Demo module not available.** Ensure you're running from the repo root and have Python path configured."))
    except Exception as e:
        display(Markdown(f"✗ **Data load failed:** {str(e)[:200]}"))
else:
    display(Markdown(f"""✓ **Data already loaded:**
- Accounts: {account_count:,}
- High risk (>70): {high_risk_count:,}
- Blocked accounts: {blocked_count:,}
- Missing openedDate: {opened_missing_count:,}
- Suspicious but unflagged: {suspicious_unflagged_count:,}"""))

# Sanity check translator support for the section 9d predicate.
check_9d = explain("g.V().hasLabel('Account').where(outE('FLAGGED_BY').count().is(0)).limit(1)")
if check_9d.get("error"):
    display(Markdown(f"⚠️ **Translator check:** section 9d predicate still unsupported — {check_9d['error']}"))


**Health:** `{"service":"graph-query-engine","status":"ok"}`    **Provider:** `tinkergraph`

**Mapping upload:** HTTP 201 — {"message":"Mapping loaded","vertexLabels":["Account","Bank","Transaction","Country","Alert"],"edgeLabels":["TRANSFER","

**Data loading:** empty graph — loading AML demo data...

✓ **Data loaded successfully:**
- Rows loaded: 100,000
- Accounts created: 81,352
- Transfers created: 100,000
- Accounts in graph: 81,352
- High risk (>70): 39,945
- Blocked accounts: 8,059
- Missing openedDate: 24,538
- Suspicious but unflagged: 0

---
## 1 — Root steps: `g.V()`, `g.E()`, `g.V(id)`

The three root forms that start every traversal.


In [3]:
show(
    "g.V().hasLabel('Account').count()",
    title="1a  g.V() — count all Account vertices",
)


### 1a  g.V() — count all Account vertices

**Gremlin:**
```groovy
g.V().hasLabel('Account').count()
```

**SQL:**
```sql
SELECT COUNT(*) AS count FROM aml_accounts
```

**Result count:** 1

1. 81352


In [4]:
show(
    "g.E().hasLabel('TRANSFER').count()",
    title="1b  g.E() — count all TRANSFER edges",
)


### 1b  g.E() — count all TRANSFER edges

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').count()
```

**SQL:**
```sql
SELECT COUNT(*) AS count FROM aml_transfers
```

**Result count:** 1

1. 100000


In [5]:
# Fetch one account ID first so the g.V(id) demo is self-contained
_first_id_res = run("g.V().hasLabel('Account').values('accountId').limit(1)")
_first_account_id = (_first_id_res.get("results") or ["ACC-001"])[0]
show(
    f"g.V().hasLabel('Account').has('accountId','{_first_account_id}').project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore')",
    title="1c  g.V(id) — look up a specific account by property",
)


### 1c  g.V(id) — look up a specific account by property

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('accountId','806758E70').project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore')
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml_accounts v WHERE v.account_id = ?
```

**Parameters:** `['806758E70']`

**Result count:** 1

,accountId,bankId,riskScore
0,806758E70,0115700,32


---
## 2 — Filter steps: `has`, `hasLabel`, `hasId`, `hasNot`, `is`


In [6]:
show(
    "g.V().hasLabel('Account').has('isBlocked', 'true').project('accountId','bankId','isBlocked').by('accountId').by('bankId').by('isBlocked').limit(10)",
    title="2a  has(k, v) — blocked accounts",
)


### 2a  has(k, v) — blocked accounts

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('isBlocked', 'true').project('accountId','bankId','isBlocked').by('accountId').by('bankId').by('isBlocked').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.is_blocked AS "isBlocked" FROM aml_accounts v WHERE v.is_blocked = ? LIMIT 10
```

**Parameters:** `['true']`

**Result count:** 10

,accountId,bankId,isBlocked
0,80E3BDCD0,016109,true
1,80675C660,01467,true
2,80675E200,0115249,true
3,8000EBAC0,001,true
4,803C038B0,017222,true
5,80675F880,013264,true
6,80E3C8650,022828,true
7,803C09680,01588,true
8,80E3ADA80,0338635,true
9,80AEF5310,0211050,true


In [7]:
show(
    "g.V().hasLabel('Account').has('riskScore', gt(70)).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').order().by(select('riskScore'), Order.desc).limit(10)",
    title="2b  has(k, gt(v)) — high risk-score accounts (predicate filter)",
)


### 2b  has(k, gt(v)) — high risk-score accounts (predicate filter)

**Gremlin:**
```groovy
g.V().hasLabel('Account').has('riskScore', gt(70)).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').order().by(select('riskScore'), Order.desc).limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml_accounts v WHERE v.risk_score > ? ORDER BY riskScore DESC LIMIT 10
```

**Parameters:** `['70']`

**Result count:** 10

,accountId,bankId,riskScore
0,80012AB70,010,173
1,80C2B9DC0,028771,173
2,80E040C50,0223635,173
3,803DC5710,023402,173
4,80E6B8060,001412,173
5,813BF0B90,0145200,173
6,80D2D54F0,009129,173
7,801B5D450,003420,173
8,80C626790,004866,173
9,8040A7BE0,028693,173


In [8]:
# hasId — filter vertex by its graph ID (typed literal)
_id_res = run("g.V().hasLabel('Account').id().limit(1)")
_vid = (_id_res.get("results") or [1])[0]
if isinstance(_vid, (int, float)) or (isinstance(_vid, str) and _vid.isdigit()):
    _vid_literal = str(_vid)
else:
    _vid_literal = f"'{_vid}'"
show(
    f"g.V().hasLabel('Account').hasId({_vid_literal}).project('accountId','bankId').by('accountId').by('bankId')",
    title="2c  hasId(id) — look up vertex by its graph ID",
)


### 2c  hasId(id) — look up vertex by its graph ID

**Gremlin:**
```groovy
g.V().hasLabel('Account').hasId(786442).project('accountId','bankId').by('accountId').by('bankId')
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE v.id = ?
```

**Parameters:** `['786442']`

**Result count:** 0

In [9]:
show(
    "g.V().hasLabel('Account').hasNot('openedDate').project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="2d  hasNot(k) — accounts with no openedDate (IS NULL)",
)


### 2d  hasNot(k) — accounts with no openedDate (IS NULL)

**Gremlin:**
```groovy
g.V().hasLabel('Account').hasNot('openedDate').project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE v.opened_date IS NULL LIMIT 10
```

**Result count:** 10

,accountId,bankId
0,80C15ED80,028255
1,8019773B0,032390
2,8067598C0,009129
3,80C163090,0115249
4,80E3BDCD0,016109
5,803BF9460,001686
6,80196EC80,011405
7,8000F5340,001
8,803C04800,006179
9,80C1A00D0,0115249


In [10]:
show(
    "g.V().hasLabel('Account').values('riskScore').is(gt(80)).limit(10)",
    title="2e  values('p').is(pred) — risk scores above 80",
)


### 2e  values('p').is(pred) — risk scores above 80

**Gremlin:**
```groovy
g.V().hasLabel('Account').values('riskScore').is(gt(80)).limit(10)
```

**SQL:**
```sql
SELECT risk_score AS riskScore FROM aml_accounts WHERE risk_score > ? LIMIT 10
```

**Parameters:** `['80']`

**Result count:** 10

1. 82
2. 122
3. 82
4. 158
5. 94
6. 154
7. 98
8. 167
9. 126
10. 86


---
## 3 — Traversal / hop steps: `out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV`


In [11]:
show(
    "g.V().hasLabel('Account').limit(5).out('BELONGS_TO').project('bankId','bankName').by('bankId').by('bankName')",
    title="3a  out(l) — Account → Bank hop",
)


### 3a  out(l) — Account → Bank hop

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).out('BELONGS_TO').project('bankId','bankName').by('bankId').by('bankName')
```

**SQL:**
```sql
SELECT v1.bank_id AS "bankId", v1.bank_name AS "bankName" FROM (SELECT * FROM aml_accounts LIMIT 5) v0 JOIN aml_account_bank e1 ON e1.account_id = v0.id JOIN aml_banks v1 ON v1.id = e1.bank_id
```

**Result count:** 5

,bankId,bankName
0,0115700,Bank-0115700
1,028255,Bank-028255
2,032390,Bank-032390
3,009129,Bank-009129
4,0115249,Bank-0115249


In [12]:
show(
    "g.V().hasLabel('Bank').limit(5).in('BELONGS_TO').project('accountId','bankId').by('accountId').by('bankId')",
    title="3b  in(l) — Bank ← Account hop",
)


### 3b  in(l) — Bank ← Account hop

**Gremlin:**
```groovy
g.V().hasLabel('Bank').limit(5).in('BELONGS_TO').project('accountId','bankId').by('accountId').by('bankId')
```

**SQL:**
```sql
SELECT v1.account_id AS "accountId", v1.bank_id AS "bankId" FROM (SELECT * FROM aml_banks LIMIT 5) v0 JOIN aml_account_bank e1 ON e1.bank_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.account_id
```

**Result count:** 2688

,accountId,bankId
0,8002B4B00,010
1,800C5F4B0,010
2,8003A8280,010
3,800CAF770,010
4,8004AAC30,010
5,809C0EA40,010
6,8009F5C50,010
7,804B424A0,010
8,80A16D180,010
9,800F9D170,010


In [13]:
show(
    "g.V().hasLabel('Account').limit(3).both('TRANSFER').dedup().project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="3c  both(l) — bidirectional TRANSFER neighbours",
)


### 3c  both(l) — bidirectional TRANSFER neighbours

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(3).both('TRANSFER').dedup().project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT DISTINCT * FROM (SELECT v1.account_id AS "accountId", v1.bank_id AS "bankId" FROM (SELECT * FROM aml_accounts LIMIT 3) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id UNION SELECT v1.account_id AS "accountId", v1.bank_id AS "bankId" FROM (SELECT * FROM aml_accounts LIMIT 3) v0 JOIN aml_transfers e1 ON e1.to_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.from_account_id) _u LIMIT 10
```

**Result count:** 3

,accountId,bankId
0,806758E70,0115700
1,808E03500,025075
2,8019773B0,032390


In [14]:
show(
    "g.V().hasLabel('Account').limit(5).outE('TRANSFER').project('transactionId','amount','currency','isLaundering').by('transactionId').by('amount').by('currency').by('isLaundering').limit(10)",
    title="3d  outE(l) — outgoing TRANSFER edges",
)


### 3d  outE(l) — outgoing TRANSFER edges

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).outE('TRANSFER').project('transactionId','amount','currency','isLaundering').by('transactionId').by('amount').by('currency').by('isLaundering').limit(10)
```

**SQL:**
```sql
SELECT e1.transaction_id AS "transactionId", e1.amount AS "amount", e1.currency AS "currency", e1.is_laundering AS "isLaundering" FROM (SELECT * FROM aml_accounts LIMIT 5) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id LIMIT 10
```

**Result count:** 5

,transactionId,amount,currency,isLaundering
0,44964,30520.11,US Dollar,0
1,14860,709.97,US Dollar,0
2,44965,1844.17,US Dollar,0
3,75214,24.76,US Dollar,0
4,75208,146.43,US Dollar,0


In [15]:
show(
    "g.V().hasLabel('Account').limit(5).inE('TRANSFER').project('transactionId','amount','currency').by('transactionId').by('amount').by('currency').limit(10)",
    title="3e  inE(l) — incoming TRANSFER edges",
)


### 3e  inE(l) — incoming TRANSFER edges

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).inE('TRANSFER').project('transactionId','amount','currency').by('transactionId').by('amount').by('currency').limit(10)
```

**SQL:**
```sql
SELECT e1.transaction_id AS "transactionId", e1.amount AS "amount", e1.currency AS "currency" FROM (SELECT * FROM aml_accounts LIMIT 5) v0 JOIN aml_transfers e1 ON e1.to_account_id = v0.id LIMIT 10
```

**Result count:** 9

,transactionId,amount,currency
0,44964,30520.11,US Dollar
1,75209,418883.46,US Dollar
2,75212,902365.29,US Dollar
3,75210,24386.26,US Dollar
4,75211,1410047.06,US Dollar
5,14860,709.97,US Dollar
6,44965,1844.17,US Dollar
7,75214,24.76,US Dollar
8,75208,146.43,US Dollar


In [16]:
show(
    "g.V().hasLabel('Account').limit(3).bothE('TRANSFER').project('transactionId','amount').by('transactionId').by('amount').limit(10)",
    title="3f  bothE(l) — all TRANSFER edges in either direction (hop step)",
)


### 3f  bothE(l) — all TRANSFER edges in either direction (hop step)

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(3).bothE('TRANSFER').project('transactionId','amount').by('transactionId').by('amount').limit(10)
```

**SQL:**
```sql
SELECT e1.transaction_id AS "transactionId", e1.amount AS "amount" FROM (SELECT * FROM aml_accounts LIMIT 3) v0 JOIN aml_transfers e1 ON (e1.from_account_id = v0.id OR e1.to_account_id = v0.id) LIMIT 10
```

**Result count:** 8

,transactionId,amount
0,44964,30520.11
1,44964,30520.11
2,75209,418883.46
3,75212,902365.29
4,75210,24386.26
5,75211,1410047.06
6,14860,709.97
7,14860,709.97


In [17]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).outV().project('accountId','bankId').by('accountId').by('bankId')",
    title="3g  outV() — source vertex of each TRANSFER edge",
)


### 3g  outV() — source vertex of each TRANSFER edge

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).outV().project('accountId','bankId').by('accountId').by('bankId')
```

**SQL:**
```sql
SELECT DISTINCT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v JOIN aml_transfers e ON v.id = e.from_account_id LIMIT 5
```

**Result count:** 5

,accountId,bankId
0,803BF3C80,016224
1,100428660,070
2,80E3BCFC0,016031
3,809594300,015730
4,80E3BC230,028827


In [18]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).inV().project('accountId','bankId').by('accountId').by('bankId')",
    title="3h  inV() — destination vertex of each TRANSFER edge",
)


### 3h  inV() — destination vertex of each TRANSFER edge

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).inV().project('accountId','bankId').by('accountId').by('bankId')
```

**SQL:**
```sql
SELECT DISTINCT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v JOIN aml_transfers e ON v.id = e.to_account_id LIMIT 5
```

**Result count:** 5

,accountId,bankId
0,803BF3C80,016224
1,809594300,015730
2,80E3BCFC0,016031
3,809594300,015730
4,80E3BC230,028827


In [19]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).bothV().dedup().project('accountId','bankId').by('accountId').by('bankId')",
    title="3i  bothV() — all endpoint vertices of TRANSFER edges (UNION of out+in)",
)


### 3i  bothV() — all endpoint vertices of TRANSFER edges (UNION of out+in)

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).bothV().dedup().project('accountId','bankId').by('accountId').by('bankId')
```

**SQL:**
```sql
SELECT DISTINCT * FROM (SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v JOIN aml_transfers e ON v.id = e.from_account_id UNION ALL SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v JOIN aml_transfers e ON v.id = e.to_account_id) _uv LIMIT 5
```

**Result count:** 5

,accountId,bankId
0,803BF3C80,016224
1,100428660,070
2,809594300,015730
3,80E3BCFC0,016031
4,80E3BC230,028827


In [20]:
show(
    "g.V().hasLabel('Account').limit(5).outE('BELONGS_TO').otherV().project('bankId','bankName').by('bankId').by('bankName')",
    title="3j  otherV() — opposite endpoint after outE",
)


### 3j  otherV() — opposite endpoint after outE

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(5).outE('BELONGS_TO').otherV().project('bankId','bankName').by('bankId').by('bankName')
```

**SQL:**
```sql
SELECT v1.bank_id AS "bankId", v1.bank_name AS "bankName" FROM (SELECT * FROM aml_accounts LIMIT 5) v0 JOIN aml_account_bank e1 ON e1.account_id = v0.id JOIN aml_banks v1 ON v1.id = e1.bank_id
```

**Result count:** 5

,bankId,bankName
0,0115700,Bank-0115700
1,028255,Bank-028255
2,032390,Bank-032390
3,009129,Bank-009129
4,0115249,Bank-0115249


---
## 4 — Repeat / path steps


In [21]:
show(
    "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(2).path().by('accountId').limit(10)",
    title="4a  repeat(out).times(n) + simplePath() — two-hop money trail",
)


### 4a  repeat(out).times(n) + simplePath() — two-hop money trail

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(2).path().by('accountId').limit(10)
```

**SQL:**
```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2 FROM (SELECT * FROM aml_accounts WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id JOIN aml_transfers e2 ON e2.from_account_id = v1.id JOIN aml_accounts v2 ON v2.id = e2.to_account_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) LIMIT 10
```

**Parameters:** `['1']`

**Result count:** 10

1. ['100428660', '806D454D0', '80E6A8770']
2. ['100428660', '80A00E360', '80CFA3160']
3. ['100428660', '80A00E360', '80CFA3160']
4. ['100428660', '80A00E360', '80CFA3160']
5. ['100428660', '80A00E360', '80CFA3160']
6. ['100428660', '80A00E360', '80CFA3160']
7. ['100428660', '80A00E360', '80CFA3160']
8. ['100428660', '80BAF4AB0', '80BAF4B00']
9. ['100428660', '80BAF4AB0', '80BAF4B00']
10. ['100428660', '80BAF4AB0', '80BAF4B00']


In [22]:
show(
    "g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)",
    title="4b  repeat with multi-label + path().by(p) — Account → Bank → Country",
)


### 4b  repeat with multi-label + path().by(p) — Account → Bank → Country

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)
```

**SQL:**
```sql
SELECT v0.account_id AS accountId0, v1.bank_name AS bankName1, v2.country_name AS countryName2 FROM (SELECT * FROM aml_accounts LIMIT 1) v0 JOIN aml_account_bank e1 ON e1.account_id = v0.id JOIN aml_banks v1 ON v1.id = e1.bank_id JOIN aml_bank_country e2 ON e2.bank_id = v1.id JOIN aml_countries v2 ON v2.id = e2.country_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) LIMIT 10
```

**Result count:** 1

1. ['806758E70', 'Bank-0115700', 'United States']


---
## 5 — Aggregation steps: `count`, `sum`, `mean`, `groupCount`


In [23]:
show(
    "g.V().hasLabel('Account').count()",
    title="5a  count() — total Account vertices",
)


### 5a  count() — total Account vertices

**Gremlin:**
```groovy
g.V().hasLabel('Account').count()
```

**SQL:**
```sql
SELECT COUNT(*) AS count FROM aml_accounts
```

**Result count:** 1

1. 81352


In [24]:
show(
    "g.V().hasLabel('Account').dedup().count()",
    title="5b  dedup() + count() → COUNT(DISTINCT id)",
)


### 5b  dedup() + count() → COUNT(DISTINCT id)

**Gremlin:**
```groovy
g.V().hasLabel('Account').dedup().count()
```

**SQL:**
```sql
SELECT COUNT(DISTINCT id) AS count FROM aml_accounts
```

**Result count:** 1

1. 81352


In [25]:
show(
    "g.E().hasLabel('TRANSFER').values('amount').sum()",
    title="5c  values('amount').sum() — total transferred amount",
)


### 5c  values('amount').sum() — total transferred amount

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').values('amount').sum()
```

**SQL:**
```sql
SELECT SUM(amount) AS sum FROM aml_transfers
```

**Result count:** 1

1. 75634082295.79884


In [26]:
show(
    "g.E().hasLabel('TRANSFER').values('amount').mean()",
    title="5d  values('amount').mean() — average transfer amount",
)


### 5d  values('amount').mean() — average transfer amount

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').values('amount').mean()
```

**SQL:**
```sql
SELECT AVG(amount) AS mean FROM aml_transfers
```

**Result count:** 1

1. 756340.8229579885


In [27]:
show(
    "g.V().hasLabel('Account').groupCount().by('bankId').order().by(values, desc).limit(10)",
    title="5e  groupCount().by(p) — accounts per bank, descending",
)


### 5e  groupCount().by(p) — accounts per bank, descending

**Gremlin:**
```groovy
g.V().hasLabel('Account').groupCount().by('bankId').order().by(values, desc).limit(10)
```

**SQL:**
```sql
SELECT bank_id AS "bankId", COUNT(*) AS count FROM aml_accounts GROUP BY bank_id ORDER BY count DESC LIMIT 10
```

**Result count:** 1

,0326103,0328767,0326105,033899,033898,033896,032565,033894,0319805,033893,...,0336937,03369,0344549,0018679,03365,0318286,0318287,0324953,0324951,031828
0,1,1,4,3,2,3,2,2,2,3,...,1,7,1,118,6,1,1,1,1,4


---
## 6 — Projection steps: `values`, `project().by()`, `valueMap`, `by(identity())`


In [28]:
show(
    "g.V().hasLabel('Account').values('accountId').limit(10)",
    title="6a  values(p) — single property column",
)


### 6a  values(p) — single property column

**Gremlin:**
```groovy
g.V().hasLabel('Account').values('accountId').limit(10)
```

**SQL:**
```sql
SELECT account_id AS accountId FROM aml_accounts LIMIT 10
```

**Result count:** 10

1. 806758E70
2. 80C15ED80
3. 8019773B0
4. 8067598C0
5. 80C163090
6. 80E3BCFC0
7. 801977590
8. 806759B30
9. 803BF4980
10. 80E3BE180


In [29]:
show(
    "g.V().hasLabel('Account').project('accountId','bankId','riskScore','accountType').by('accountId').by('bankId').by('riskScore').by('accountType').limit(10)",
    title="6b  project().by(p) — multi-column SELECT",
)


### 6b  project().by(p) — multi-column SELECT

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','bankId','riskScore','accountType').by('accountId').by('bankId').by('riskScore').by('accountType').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore", v.account_type AS "accountType" FROM aml_accounts v LIMIT 10
```

**Result count:** 10

,accountId,bankId,riskScore,accountType
0,806758E70,0115700,32,CORPORATE
1,80C15ED80,028255,75,CORPORATE
2,8019773B0,032390,41,PERSONAL
3,8067598C0,009129,22,CORPORATE
4,80C163090,0115249,82,CORPORATE
5,80E3BCFC0,016031,46,CORPORATE
6,801977590,021174,42,CORPORATE
7,806759B30,0017246,20,CORPORATE
8,803BF4980,007478,122,CORPORATE
9,80E3BE180,0038486,82,CORPORATE


In [30]:
show(
    "g.V().hasLabel('Account').valueMap().limit(5)",
    title="6c  valueMap() — all mapped properties as a map (root vertex)",
)


### 6c  valueMap() — all mapped properties as a map (root vertex)

**Gremlin:**
```groovy
g.V().hasLabel('Account').valueMap().limit(5)
```

**SQL:**
```sql
SELECT account_id AS "accountId", bank_id AS "bankId", account_type AS "accountType", risk_score AS "riskScore", is_blocked AS "isBlocked", opened_date AS "openedDate" FROM aml_accounts LIMIT 5
```

**Result count:** 5

,openedDate,accountId,bankId,accountType,isBlocked,riskScore
0,[2020-01-01],[806758E70],[0115700],[CORPORATE],[false],[32]
1,NaN,[80C15ED80],[028255],[CORPORATE],[false],[75]
2,NaN,[8019773B0],[032390],[PERSONAL],[false],[41]
3,NaN,[8067598C0],[009129],[CORPORATE],[false],[22]
4,NaN,[80C163090],[0115249],[CORPORATE],[false],[82]


In [31]:
show(
    "g.E().hasLabel('TRANSFER').valueMap().limit(5)",
    title="6d  valueMap() — all mapped properties for edges",
)


### 6d  valueMap() — all mapped properties for edges

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').valueMap().limit(5)
```

**SQL:**
```sql
SELECT transaction_id AS "transactionId", amount AS "amount", currency AS "currency", payment_format AS "paymentFormat", event_time AS "eventTime", is_laundering AS "isLaundering" FROM aml_transfers LIMIT 5
```

**Result count:** 5

,amount,paymentFormat,eventTime,currency,isLaundering,transactionId
0,4.17,Reinvestment,2022/09/01 00:17,US Dollar,0,29851
1,32625.40,Cash,2022/09/01 00:29,US Dollar,0,60091
2,89946.80,Reinvestment,2022/09/01 00:12,US Dollar,0,90407
3,6.54,Reinvestment,2022/09/01 00:26,US Dollar,0,60092
4,18.02,Reinvestment,2022/09/01 00:22,US Dollar,0,90406


In [32]:
show(
    "g.V().hasLabel('Account').project('id','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(10)",
    title="6e  by(identity()) in project() — emits the vertex/element id",
)


### 6e  by(identity()) in project() — emits the vertex/element id

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('id','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.id AS "id", v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v LIMIT 10
```

**Result count:** 10

,id,accountId,bankId
0,"{'id': 786442, 'label': 'Account', 'properties...",806758E70,0115700
1,"{'id': 1310745, 'label': 'Account', 'propertie...",80C15ED80,028255
2,"{'id': 262165, 'label': 'Account', 'properties...",8019773B0,032390
3,"{'id': 786461, 'label': 'Account', 'properties...",8067598C0,009129
4,"{'id': 1310727, 'label': 'Account', 'propertie...",80C163090,0115249
5,"{'id': 1572878, 'label': 'Account', 'propertie...",80E3BCFC0,016031
6,"{'id': 262183, 'label': 'Account', 'properties...",801977590,021174
7,"{'id': 786479, 'label': 'Account', 'properties...",806759B30,0017246
8,"{'id': 524322, 'label': 'Account', 'properties...",803BF4980,007478
9,"{'id': 1572915, 'label': 'Account', 'propertie...",80E3BE180,0038486


In [33]:
show(
    "g.V().hasLabel('Account').limit(10).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())",
    title="6f  by(out(l).values(p).fold()) — neighbour property list per vertex",
)


### 6f  by(out(l).values(p).fold()) — neighbour property list per vertex

**Gremlin:**
```groovy
g.V().hasLabel('Account').limit(10).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT STRING_AGG(_njv0.bank_name, ',') FROM aml_account_bank _nje0 JOIN aml_banks _njv0 ON _njv0.id = _nje0.bank_id WHERE _nje0.account_id = v.id) AS "bankName" FROM aml_accounts v LIMIT 10
```

**Result count:** 10

,accountId,bankId,bankName
0,806758E70,0115700,[Bank-0115700]
1,80C15ED80,028255,[Bank-028255]
2,8019773B0,032390,[Bank-032390]
3,8067598C0,009129,[Bank-009129]
4,80C163090,0115249,[Bank-0115249]
5,80E3BCFC0,016031,[Bank-016031]
6,801977590,021174,[Bank-021174]
7,806759B30,0017246,[Bank-0017246]
8,803BF4980,007478,[Bank-007478]
9,80E3BE180,0038486,[Bank-0038486]


In [34]:
show(
    "g.V().hasLabel('Account').project('accountId','outDegree','inDegree').by('accountId').by(outE('TRANSFER').count()).by(inE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)",
    title="6g  by(outE/inE.count()) — edge-degree subqueries in project()",
)


### 6g  by(outE/inE.count()) — edge-degree subqueries in project()

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','outDegree','inDegree').by('accountId').by(outE('TRANSFER').count()).by(inE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id) AS "outDegree", (SELECT COUNT(*) FROM aml_transfers WHERE to_account_id = v.id) AS "inDegree" FROM aml_accounts v ORDER BY outDegree DESC LIMIT 10
```

**Result count:** 10

,accountId,outDegree,inDegree
0,100428660,1924,2
1,8001409E0,13,2
2,80026D340,13,0
3,800631890,13,0
4,8001C3570,12,0
5,8008F0D20,12,3
6,8001523C0,11,2
7,8001A4930,11,1
8,8001F3760,11,2
9,800537DE0,11,2


In [35]:
show(
    "g.V().hasLabel('Account').project('accountId','riskBucket').by('accountId').by(choose(values('riskScore').is(gte(70)), constant('HIGH'), constant('NORMAL'))).limit(15)",
    title="6h  by(choose(values.is, constant, constant)) — CASE WHEN projection",
)


### 6h  by(choose(values.is, constant, constant)) — CASE WHEN projection

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','riskBucket').by('accountId').by(choose(values('riskScore').is(gte(70)), constant('HIGH'), constant('NORMAL'))).limit(15)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", CASE WHEN v.risk_score >= 70 THEN 'HIGH' ELSE 'NORMAL' END AS "riskBucket" FROM aml_accounts v LIMIT 15
```

**Result count:** 15

,accountId,riskBucket
0,806758E70,NORMAL
1,80C15ED80,HIGH
2,8019773B0,NORMAL
3,8067598C0,NORMAL
4,80C163090,HIGH
5,80E3BCFC0,NORMAL
6,801977590,NORMAL
7,806759B30,NORMAL
8,803BF4980,HIGH
9,80E3BE180,HIGH


In [36]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).project('fromAccountId','toAccountId','amount').by(outV().values('accountId')).by(inV().values('accountId')).by('amount')",
    title="6i  by(outV/inV.values(p)) — edge project with endpoint properties",
)


### 6i  by(outV/inV.values(p)) — edge project with endpoint properties

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').limit(5).project('fromAccountId','toAccountId','amount').by(outV().values('accountId')).by(inV().values('accountId')).by('amount')
```

**SQL:**
```sql
SELECT ov.account_id AS "fromAccountId", iv.account_id AS "toAccountId", e.amount AS "amount" FROM aml_transfers e JOIN aml_accounts ov ON ov.id = e.from_account_id JOIN aml_accounts iv ON iv.id = e.to_account_id LIMIT 5
```

**Result count:** 5

,fromAccountId,toAccountId,amount
0,803BF3C80,803BF3C80,4.17
1,100428660,809594300,32625.40
2,80E3BCFC0,80E3BCFC0,89946.80
3,809594300,809594300,6.54
4,80E3BC230,80E3BC230,18.02


---
## 7 — Ordering and paging: `order().by`, `limit`, `dedup`


In [37]:
show(
    "g.V().hasLabel('Account').order().by('riskScore', Order.desc).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)",
    title="7a  order().by(p, desc) — top-risk accounts",
)


### 7a  order().by(p, desc) — top-risk accounts

**Gremlin:**
```groovy
g.V().hasLabel('Account').order().by('riskScore', Order.desc).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml_accounts v ORDER BY riskScore DESC LIMIT 10
```

**Result count:** 10

,accountId,bankId,riskScore
0,80012AB70,010,173
1,80C2B9DC0,028771,173
2,80E040C50,0223635,173
3,803DC5710,023402,173
4,80E6B8060,001412,173
5,813BF0B90,0145200,173
6,80D2D54F0,009129,173
7,801B5D450,003420,173
8,80C626790,004866,173
9,8040A7BE0,028693,173


In [38]:
show(
    "g.V().hasLabel('Account').project('accountId','outDegree').by('accountId').by(outE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)",
    title="7b  order().by(select(a), desc) — order by projected alias",
)


### 7b  order().by(select(a), desc) — order by projected alias

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','outDegree').by('accountId').by(outE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id) AS "outDegree" FROM aml_accounts v ORDER BY outDegree DESC LIMIT 10
```

**Result count:** 10

,accountId,outDegree
0,100428660,1924
1,8001409E0,13
2,80026D340,13
3,800631890,13
4,8001C3570,12
5,8008F0D20,12
6,8001523C0,11
7,8001A4930,11
8,8001F3760,11
9,800537DE0,11


In [39]:
show(
    "g.V().hasLabel('Account').dedup().count()",
    title="7c  dedup() — distinct accounts (COUNT DISTINCT)",
)


### 7c  dedup() — distinct accounts (COUNT DISTINCT)

**Gremlin:**
```groovy
g.V().hasLabel('Account').dedup().count()
```

**SQL:**
```sql
SELECT COUNT(DISTINCT id) AS count FROM aml_accounts
```

**Result count:** 1

1. 81352


---
## 8 — Alias / select / where steps


In [40]:
show(
    "g.V().hasLabel('Account').as('a').out('TRANSFER').as('b').where('a', neq('b')).select('a','b').by('accountId').by('accountId').limit(10)",
    title="8a  as(l) + where(a, neq(b)) + select().by(p) — self-loop exclusion",
)


### 8a  as(l) + where(a, neq(b)) + select().by(p) — self-loop exclusion

**Gremlin:**
```groovy
g.V().hasLabel('Account').as('a').out('TRANSFER').as('b').where('a', neq('b')).select('a','b').by('accountId').by('accountId').limit(10)
```

**SQL:**
```sql
SELECT v1.* FROM aml_accounts v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id WHERE v0.id <> v1.id LIMIT 10
```

**Result count:** 10

,a,b
0,806754D20,8016CDA50
1,8000F4580,8000F5340
2,8001F4270,80675C660
3,80196F3B0,80F5771E0
4,80A3B6A10,80C19FEB0
5,80A3B6A10,80C19FEB0
6,80E3BF540,80024DDD0
7,8095F1A70,80ED0D260
8,8095F1A70,80ED0D260
9,8095F1A70,80ED0D260


In [41]:
show(
    "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="8b  where(outE(l).has(k,v)) — correlated EXISTS subquery",
)


### 8b  where(outE(l).has(k,v)) — correlated EXISTS subquery

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = v.id AND we.is_laundering = ?) LIMIT 10
```

**Parameters:** `['1']`

**Result count:** 1

,accountId,bankId
0,100428660,070


In [42]:
show(
    "g.V().hasLabel('Account').where(inE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="8c  where(inE(l).has(k,v)) — correlated EXISTS on incoming edge",
)


### 8c  where(inE(l).has(k,v)) — correlated EXISTS on incoming edge

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(inE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.to_account_id = v.id AND we.is_laundering = ?) LIMIT 10
```

**Parameters:** `['1']`

**Result count:** 5

,accountId,bankId
0,80E480620,0032375
1,800825340,001124
2,80B39E7B0,015980
3,80DC756E0,0113798
4,805B716C0,011474


In [43]:
show(
    "g.V().hasLabel('Account').where(bothE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="8d  where(bothE(l).has(k,v)) — EXISTS on either direction",
)


### 8d  where(bothE(l).has(k,v)) — EXISTS on either direction

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(bothE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE (we.from_account_id = v.id OR we.to_account_id = v.id) AND we.is_laundering = ?) LIMIT 10
```

**Parameters:** `['1']`

**Result count:** 6

,accountId,bankId
0,100428660,070
1,80E480620,0032375
2,800825340,001124
3,80B39E7B0,015980
4,80DC756E0,0113798
5,805B716C0,011474


In [44]:
show(
    "g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)",
    title="8e  where(out(l).has(k,v)) — correlated EXISTS with vertex JOIN",
)


### 8e  where(out(l).has(k,v)) — correlated EXISTS with vertex JOIN

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_transfer_channel _we JOIN aml_countries _wv ON _wv.id = _we.country_id WHERE _we.transfer_id = v.id AND _wv.fatf_blacklist = ?) LIMIT 10
```

**Parameters:** `['true']`

**Result count:** 10

,accountId,bankId,riskScore
0,8067598C0,009129,22
1,80C163090,0115249,82
2,80E3BCFC0,016031,46
3,80925E3C0,0221891,42
4,801977D30,01665,45
5,803BF9460,001686,154
6,80196EC80,011405,98
7,803C029A0,027444,126
8,80675C660,01467,17
9,80196F310,011405,70


In [45]:
show(
    "g.V().hasLabel('Account').project('accountId','suspOut').by('accountId').by(outE('TRANSFER').has('isLaundering','1').count()).where(select('suspOut').is(gt(0))).order().by(select('suspOut'), Order.desc).limit(10)",
    title="8f  where(select(a).is(gt(n))) — HAVING-style filter on projected alias",
)


### 8f  where(select(a).is(gt(n))) — HAVING-style filter on projected alias

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('accountId','suspOut').by('accountId').by(outE('TRANSFER').has('isLaundering','1').count()).where(select('suspOut').is(gt(0))).order().by(select('suspOut'), Order.desc).limit(10)
```

**SQL:**
```sql
SELECT * FROM (SELECT v.account_id AS "accountId", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id AND is_laundering = '1') AS "suspOut" FROM aml_accounts v) p WHERE p."suspOut" > 0 ORDER BY suspOut DESC LIMIT 10
```

**Result count:** 1

,accountId,suspOut
0,100428660,5


---
## 9 — Compound `where`: `and`, `or`, `not`


In [46]:
show(
    "g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY'))).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="9a  where(and(p1, p2)) — accounts that are both suspicious AND flagged",
)


### 9a  where(and(p1, p2)) — accounts that are both suspicious AND flagged

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY'))).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE ((EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = v.id AND we.is_laundering = ?)) AND (EXISTS (SELECT 1 FROM aml_account_alert we WHERE we.account_id = v.id))) LIMIT 10
```

**Parameters:** `['1']`

**Result count:** 1

,accountId,bankId
0,100428660,070


In [47]:
show(
    "g.V().hasLabel('Account').where(or(out('SENT_VIA').has('fatfBlacklist','true'), outE('TRANSFER').has('isLaundering','1'))).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)",
    title="9b  where(or(p1, p2)) — accounts linked to blacklisted country OR suspicious transfer",
)


### 9b  where(or(p1, p2)) — accounts linked to blacklisted country OR suspicious transfer

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(or(out('SENT_VIA').has('fatfBlacklist','true'), outE('TRANSFER').has('isLaundering','1'))).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml_accounts v WHERE ((EXISTS (SELECT 1 FROM aml_transfer_channel _we JOIN aml_countries _wv ON _wv.id = _we.country_id WHERE _we.transfer_id = v.id AND _wv.fatf_blacklist = ?)) OR (EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = v.id AND we.is_laundering = ?))) LIMIT 10
```

**Parameters:** `['true', '1']`

**Result count:** 10

,accountId,bankId,riskScore
0,8067598C0,009129,22
1,80C163090,0115249,82
2,80E3BCFC0,016031,46
3,80925E3C0,0221891,42
4,801977D30,01665,45
5,803BF9460,001686,154
6,80196EC80,011405,98
7,803C029A0,027444,126
8,80675C660,01467,17
9,80196F310,011405,70


In [48]:
show(
    "g.V().hasLabel('Account').where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="9c  where(outE.count().is(0)) — accounts with NO alert flags",
)


### 9c  where(outE.count().is(0)) — accounts with NO alert flags

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE (SELECT COUNT(*) FROM aml_account_alert we WHERE we.account_id = v.id) = ? LIMIT 10
```

**Parameters:** `['0']`

**Result count:** 10

,accountId,bankId
0,806758E70,0115700
1,80C15ED80,028255
2,8019773B0,032390
3,8067598C0,009129
4,80C163090,0115249
5,80E3BCFC0,016031
6,801977590,021174
7,806759B30,0017246
8,803BF4980,007478
9,80E3BE180,0038486


In [49]:
show(
    "g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="9d  where(and(p1, p2)) — suspicious but not yet flagged",
)


### 9d  where(and(p1, p2)) — suspicious but not yet flagged

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId').by('accountId').by('bankId').limit(10)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v WHERE ((EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = v.id AND we.is_laundering = ?)) AND ((SELECT COUNT(*) FROM aml_account_alert we WHERE we.account_id = v.id) = ?)) LIMIT 10
```

**Parameters:** `['1', '0']`

**Result count:** 0

---
## 10 — `groupCount` with alias key


In [50]:
show(
    "g.V().hasLabel('Account').as('a').out('BELONGS_TO').as('b').groupCount().by(select('a').by('bankId')).order().by(values, desc).limit(10)",
    title="10a  groupCount().by(select(a).by(p)) — accounts per bank via alias key",
)


### 10a  groupCount().by(select(a).by(p)) — accounts per bank via alias key

**Gremlin:**
```groovy
g.V().hasLabel('Account').as('a').out('BELONGS_TO').as('b').groupCount().by(select('a').by('bankId')).order().by(values, desc).limit(10)
```

**SQL:**
```sql
SELECT v0.bank_id AS "bankId", COUNT(*) AS count FROM aml_accounts v0 JOIN aml_account_bank e1 ON e1.account_id = v0.id JOIN aml_banks v1 ON v1.id = e1.bank_id GROUP BY v0.bank_id ORDER BY count DESC LIMIT 10
```

**Result count:** 1

,0326103,0328767,0326105,033899,033898,033896,032565,033894,0319805,033893,...,0336937,03369,0344549,0018679,03365,0318286,0318287,0324953,0324951,031828
0,1,1,4,3,2,3,2,2,2,3,...,1,7,1,118,6,1,1,1,1,4


---
## 11 — Edge-root projections with `outV` / `inV`


In [51]:
show(
    "g.E().hasLabel('TRANSFER').has('isLaundering','1').project('fromAcct','toAcct','amount','currency').by(outV().values('accountId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)",
    title="11a  g.E().project(by(outV.values), by(inV.values)) — suspicious transfer endpoints",
)


### 11a  g.E().project(by(outV.values), by(inV.values)) — suspicious transfer endpoints

**Gremlin:**
```groovy
g.E().hasLabel('TRANSFER').has('isLaundering','1').project('fromAcct','toAcct','amount','currency').by(outV().values('accountId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)
```

**SQL:**
```sql
SELECT ov.account_id AS "fromAcct", iv.account_id AS "toAcct", e.amount AS "amount", e.currency AS "currency" FROM aml_transfers e JOIN aml_accounts ov ON ov.id = e.from_account_id JOIN aml_accounts iv ON iv.id = e.to_account_id WHERE e.is_laundering = ? LIMIT 15
```

**Parameters:** `['1']`

**Result count:** 5

,fromAcct,toAcct,amount,currency
0,100428660,80E480620,14288.83,US Dollar
1,100428660,800825340,389769.39,US Dollar
2,100428660,80B39E7B0,792.92,US Dollar
3,100428660,805B716C0,29024.33,US Dollar
4,100428660,80DC756E0,13171425.53,US Dollar


---
## 12 — `simplePath` across multi-hop traversals


In [52]:
show(
    "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)",
    title="12a  simplePath() — 3-hop money trail with cycle exclusion",
)


### 12a  simplePath() — 3-hop money trail with cycle exclusion

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)
```

**SQL:**
```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3 FROM (SELECT * FROM aml_accounts WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id JOIN aml_transfers e2 ON e2.from_account_id = v1.id JOIN aml_accounts v2 ON v2.id = e2.to_account_id JOIN aml_transfers e3 ON e3.from_account_id = v2.id JOIN aml_accounts v3 ON v3.id = e3.to_account_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) LIMIT 10
```

**Parameters:** `['1']`

**Result count:** 10

1. ['100428660', '80BAF4AB0', '80BAF4B00', '80A5A37B0']
2. ['100428660', '80BAF4AB0', '80BAF4B00', '80A5A37B0']
3. ['100428660', '80BAF4AB0', '80BAF4B00', '80A5A37B0']
4. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
5. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
6. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
7. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
8. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
9. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
10. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']


---
## 13 — `identity()` as no-op step and in `project`


In [53]:
show(
    "g.V().hasLabel('Account').identity().project('accountId','bankId').by('accountId').by('bankId').limit(5)",
    title="13a  identity() as a no-op mid-traversal step",
)


### 13a  identity() as a no-op mid-traversal step

**Gremlin:**
```groovy
g.V().hasLabel('Account').identity().project('accountId','bankId').by('accountId').by('bankId').limit(5)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v LIMIT 5
```

**Result count:** 5

,accountId,bankId
0,806758E70,0115700
1,80C15ED80,028255
2,8019773B0,032390
3,8067598C0,009129
4,80C163090,0115249


In [54]:
show(
    "g.V().hasLabel('Account').project('graphId','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(5)",
    title="13b  project(...).by(identity()) — emit the graph element ID",
)


### 13b  project(...).by(identity()) — emit the graph element ID

**Gremlin:**
```groovy
g.V().hasLabel('Account').project('graphId','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(5)
```

**SQL:**
```sql
SELECT v.id AS "graphId", v.account_id AS "accountId", v.bank_id AS "bankId" FROM aml_accounts v LIMIT 5
```

**Result count:** 5

,graphId,accountId,bankId
0,"{'id': 786442, 'label': 'Account', 'properties...",806758E70,0115700
1,"{'id': 1310745, 'label': 'Account', 'propertie...",80C15ED80,028255
2,"{'id': 262165, 'label': 'Account', 'properties...",8019773B0,032390
3,"{'id': 786461, 'label': 'Account', 'properties...",8067598C0,009129
4,"{'id': 1310727, 'label': 'Account', 'propertie...",80C163090,0115249


---
## 14 — Full AML showcase query (combines multiple features)


In [55]:
show(
    (
        "g.V().hasLabel('Account')"
        ".where(and("
        "  outE('TRANSFER').has('isLaundering','1'),"
        "  outE('FLAGGED_BY').count().is(0)"
        "))"
        ".project('accountId','bankId','riskScore','suspiciousOut','totalOut')"
        ".by('accountId')"
        ".by('bankId')"
        ".by('riskScore')"
        ".by(outE('TRANSFER').has('isLaundering','1').count())"
        ".by(outE('TRANSFER').count())"
        ".order().by(select('suspiciousOut'), Order.desc)"
        ".limit(20)"
    ),
    title="14  Full AML showcase — suspicious-but-unflagged accounts with degrees",
    limit=20,
)


### 14  Full AML showcase — suspicious-but-unflagged accounts with degrees

**Gremlin:**
```groovy
g.V().hasLabel('Account').where(and(  outE('TRANSFER').has('isLaundering','1'),  outE('FLAGGED_BY').count().is(0))).project('accountId','bankId','riskScore','suspiciousOut','totalOut').by('accountId').by('bankId').by('riskScore').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).order().by(select('suspiciousOut'), Order.desc).limit(20)
```

**SQL:**
```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id AND is_laundering = '1') AS "suspiciousOut", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id) AS "totalOut" FROM aml_accounts v WHERE ((EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = v.id AND we.is_laundering = ?)) AND ((SELECT COUNT(*) FROM aml_account_alert we WHERE we.account_id = v.id) = ?)) ORDER BY suspiciousOut DESC LIMIT 20
```

**Parameters:** `['1', '0']`

**Result count:** 0

---

## Summary

| Section | Steps demonstrated |
|---------|--------------------|
| 1 | `g.V()`, `g.E()`, `g.V(id)` — root steps |
| 2 | `has`, `hasLabel`, `hasId`, `hasNot`, `is(pred)` — filter steps |
| 3 | `out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV` — hop steps |
| 4 | `repeat().times(n)`, `simplePath()`, `path().by(p)` — path steps |
| 5 | `count`, `sum`, `mean`, `groupCount().by(p)`, `dedup` — aggregation |
| 6 | `values(p)`, `project().by(p)`, `valueMap()`, `by(identity())`, `by(out.values.fold)`, `by(outE/inE.count)`, `by(choose)`, `by(outV/inV.values)` — projection |
| 7 | `order().by(p, asc/desc)`, `order().by(select(a))`, `limit(n)` — ordering/paging |
| 8 | `as`, `select().by`, `where(neq)`, `where(outE/inE/bothE.has)`, `where(out.has)`, `where(select.is(gt))` — alias/where |
| 9 | `where(and(...))`, `where(or(...))`, `where(not(...))` — compound predicates |
| 10 | `groupCount().by(select(a).by(p))` — alias-keyed grouping |
| 11 | `g.E().project(by(outV.values), by(inV.values))` — edge-root projections |
| 12 | `simplePath()` across multi-hop `repeat` |
| 13 | `identity()` as no-op and in `project().by(identity())` |
| 14 | Full combined query |

All queries in sections 1–13 are translated to SQL by `/query/explain`.
Native execution (`/gremlin/query`) is used to show live results.
